<a href="https://colab.research.google.com/github/CodeCrusaders26/privacy-dns-filter/blob/feature%2Fml-model/Samuel_Dataset_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

import joblib

print("Environment ready!")

Environment ready!


In [ ]:
data = {
    "domain": [
        "google.com",
        "youtube.com",
        "wikipedia.org",
        "github.com",
        "amazon.com",
        "microsoft.com",
        "facebook.com",
        "apple.com",
        "secure-login-verify123.xyz",
        "free-prize-winner.click",
        "account-update-now.top",
        "paypal-security-check.info",
        "verify-bank-account.xyz",
        "login-confirmation123.site",
        "claim-your-reward.click",
        "update-password-now.top"
    ],
    "label": [
        0, 0, 0, 0, 0, 0, 0, 0,
        1, 1, 1, 1, 1, 1, 1, 1
    ]
}

df = pd.DataFrame(data)

print(df.head())
print("\nDataset shape:", df.shape)
print("\nLabel distribution:")
print(df["label"].value_counts())

          domain  label
0     google.com      0
1    youtube.com      0
2  wikipedia.org      0
3     github.com      0
4     amazon.com      0

Dataset shape: (16, 2)

Label distribution:
label
0    8
1    8
Name: count, dtype: int64


In [ ]:
def extract_features(domain):

    return {
        "length": len(domain),
        "dots": domain.count("."),
        "hyphens": domain.count("-"),
        "digits": sum(c.isdigit() for c in domain),
        "special_chars": sum(
            not c.isalnum() and c != "." and c != "-"
            for c in domain
        ),
        "subdomains": max(0, domain.count(".") - 1)
    }


features = pd.DataFrame(
    df["domain"].apply(extract_features).tolist()
)

print(features.head())

   length  dots  hyphens  digits  special_chars  subdomains
0      10     1        0       0              0           0
1      11     1        0       0              0           0
2      13     1        0       0              0           0
3      10     1        0       0              0           0
4      10     1        0       0              0           0


In [ ]:
X = features
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

print("Model training completed!")

Model training completed!


In [ ]:
predictions = model.predict(X_test)

print("Accuracy:",
      accuracy_score(y_test, predictions))

print("\nClassification Report:")
print(classification_report(y_test, predictions))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, predictions))

Accuracy: 1.0

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         2
           1       1.00      1.00      1.00         2

    accuracy                           1.00         4
   macro avg       1.00      1.00      1.00         4
weighted avg       1.00      1.00      1.00         4


Confusion Matrix:
[[2 0]
 [0 2]]


In [ ]:
from google.colab import files

uploaded = files.upload()

print("Uploaded:")
for f in uploaded:
    print(f)

KeyboardInterrupt: 

In [ ]:
import pandas as pd
from io import StringIO

# 1. Tranco
tranco = pd.read_csv(
    "/content/drive/MyDrive/Colab Notebooks/Dataset/top-1m.csv",
    header=None,
    names=["rank", "domain"]
)

# Take the top 10,000 candidates
tranco = tranco.head(10000)

# 2. URLhaus
with open("/content/drive/MyDrive/Colab Notebooks/Dataset/csv.txt", "r", encoding="utf-8-sig") as f:
    lines = f.readlines()

header_index = next(
    i for i, line in enumerate(lines)
    if line.lstrip("# ").startswith("id,dateadded,url,")
)

urlhaus_text = (
    lines[header_index].lstrip("# ")
    + "".join(lines[header_index + 1:])
)

urlhaus = pd.read_csv(StringIO(urlhaus_text))

# 3. PhishTank
phishtank = pd.read_csv(
    "/content/drive/MyDrive/Colab Notebooks/Dataset/verified_online.csv",
    encoding="utf-8-sig"
)

# Display columns
print("TRANCO:", tranco.columns.tolist())
print("URLHAUS:", urlhaus.columns.tolist())
print("PHISHTANK:", phishtank.columns.tolist())

print("\nDataset sizes:")
print("Tranco:", len(tranco))
print("URLhaus:", len(urlhaus))
print("PhishTank:", len(phishtank))

TRANCO: ['rank', 'domain']
URLHAUS: ['id', 'dateadded', 'url', 'url_status', 'last_online', 'threat', 'tags', 'urlhaus_link', 'reporter']
PHISHTANK: ['phish_id', 'url', 'phish_detail_url', 'submission_time', 'verified', 'verification_time', 'online', 'target']

Dataset sizes:
Tranco: 10000
URLhaus: 168557
PhishTank: 46808


In [ ]:
import pandas as pd
import re
import ipaddress
from urllib.parse import urlsplit
from datetime import date

def extract_domain(value):
    if pd.isna(value):
        return None

    value = str(value).strip()

    try:
        if "://" not in value:
            value = "//" + value

        host = urlsplit(value).hostname

        if not host:
            return None

        host = host.lower().rstrip(".")

        # Exclude IP addresses
        try:
            ipaddress.ip_address(host)
            return None
        except ValueError:
            pass

        # Validate hostname
        host = host.encode("idna").decode("ascii")

        if len(host) > 253:
            return None

        if not re.fullmatch(
            r"(?=.{1,253}$)(?:[a-z0-9](?:[a-z0-9-]{0,61}[a-z0-9])?\.)+[a-z]{2,63}",
            host
        ):
            return None

        return host

    except (ValueError, UnicodeError):
        return None

In [ ]:
safe = tranco[["domain"]].copy()

safe["domain"] = safe["domain"].apply(extract_domain)
safe = safe.dropna(subset=["domain"])

safe["label"] = 0
safe["source"] = "Tranco"

safe = safe.drop_duplicates(subset=["domain"])

print("Legitimate domains:", len(safe))

Legitimate domains: 9999


In [ ]:
# URLhaus
uh = urlhaus[["url"]].copy()

uh["domain"] = uh["url"].apply(extract_domain)
uh = uh.dropna(subset=["domain"])

uh["label"] = 1
uh["source"] = "URLhaus"

# PhishTank: retain verified phishing records
pt = phishtank[
    phishtank["verified"].astype(str).str.lower().eq("yes")
][["url"]].copy()

pt["domain"] = pt["url"].apply(extract_domain)
pt = pt.dropna(subset=["domain"])

pt["label"] = 1
pt["source"] = "PhishTank"

# Combine malicious sources
malicious = pd.concat([
    uh[["domain", "label", "source"]],
    pt[["domain", "label", "source"]]
], ignore_index=True)

# Combine sources for domains appearing in both databases
malicious = (
    malicious.groupby("domain")
    .agg(
        label=("label", "max"),
        source=("source", lambda x: "+".join(sorted(set(x))))
    )
    .reset_index()
)

print("Unique malicious domains:", len(malicious))

Unique malicious domains: 21644


In [ ]:
# Remove any malicious domains from legitimate candidates
safe = safe[~safe["domain"].isin(malicious["domain"])]

# Select up to 10,000 malicious domains
malicious_sample = malicious.sample(
    n=min(len(malicious), 10000),
    random_state=42
)

# Combine
combined = pd.concat(
    [safe, malicious_sample],
    ignore_index=True
)

combined["collection_date"] = date.today().isoformat()

# Shuffle records
combined = combined.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

# Export
combined.to_csv("combined_domains.csv", index=False)

print("Final dataset:", len(combined))
print("\nClass distribution:")
print(combined["label"].value_counts())

print("\nSample:")
print(combined.head(10))

files.download("combined_domains.csv")

Final dataset: 19950

Class distribution:
label
1    10000
0     9950
Name: count, dtype: int64

Sample:
                           domain  label     source collection_date
0                  findmykids.org      0     Tranco      2026-11-11
1                     labs.google      0     Tranco      2026-11-11
2  integrating.smbc-flexibler.com      1  PhishTank      2026-11-11
3                         unl.edu      0     Tranco      2026-11-11
4            ggi.dietondemand.com      1    URLhaus      2026-11-11
5                          bnf.fr      0     Tranco      2026-11-11
6                      qualys.com      0     Tranco      2026-11-11
7                   t.yesware.com      1  PhishTank      2026-11-11
8                   tagesschau.de      0     Tranco      2026-11-11
9                   flashscore.ua      0     Tranco      2026-11-11


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>